# Fine-Tuning: Qwen2.5-1.5B-Instruct auf den Encounter-Agent (Lauf 2)

LoRA/QLoRA-Fine-Tuning von `Qwen/Qwen2.5-1.5B-Instruct`, damit es Groqs `gpt-oss-20b`-Rolle im Encounter-Agent
uebernehmen kann (siehe `training/MODEL_CHOICE.md` fuer die Modellwahl, das LoRA-Schaltplan-Artifact fuer die
Konzepte dahinter).

**Bevor du loslegst**: `Runtime` -> `Change runtime type` -> `T4 GPU` einstellen, sonst laufen die
4-Bit-Zellen unten nicht. Dann `Runtime` -> `Run all`.

## Warum es einen Lauf 2 gibt

Lauf 1 (Attention-only, 3 Epochen, Loss ueber die ganze Sequenz) lieferte lokal nur ~23-36 % gueltige Antworten
(Groq: ~99 %, unveraendertes Basismodell: 0 %). Fast immer fehlte das letzte JSON-Feld `attack_name`.
Ausgeschlossen wurden Quantisierung, Prompt-Template, Trainingsdaten und Temperatur. Uebrig blieb Underfitting.
Lauf 2 aendert deshalb genau die Trainingsseite:

- **Loss nur auf der Antwort** (Prompt/Completion-Format) - in Lauf 1 zaehlte der fast konstante Prompt mit
- **5 statt 3 Epochen**, Early Stopping mit Geduld 2 (Val-Loss sank in Lauf 1 bis zuletzt)
- **LoRA auf allen linearen Schichten** statt nur Attention (Rang 16, Alpha 32, Dropout 0.05 unveraendert)
- **Schnelltest am Ende**: Anteil gueltiger Antworten auf den 40 Val-Prompts, damit du vor dem Download siehst,
  ob es geholfen hat

Der Prompt bleibt unveraendert. Train/Val/Test bereits in Schritt 3 gesplittet (217/40/39), auf Setting-Ebene
getrennt. Achtung: der Eval-Loss ist jetzt ein reiner Antwort-Loss und mit dem aus Lauf 1 nicht vergleichbar.

## 1. Setup

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets torchao

In [ ]:
# Sollte eine Tesla T4 mit ~15GB VRAM zeigen. Wenn nicht: Runtime -> Change runtime type -> T4 GPU.
!nvidia-smi

## 2. Trainingsdaten hochladen

Lade die drei Dateien aus `training/artifacts/dataset/` auf deinem Rechner hoch: `train.jsonl`, `val.jsonl`,
`test.jsonl`. Die liegen dort nur lokal (siehe `training/README.md` - absichtlich nicht im Git-Repo).

In [ ]:
from google.colab import files

uploaded = files.upload()  # train.jsonl, val.jsonl, test.jsonl auswählen

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": "train.jsonl", "validation": "val.jsonl", "test": "test.jsonl"},
)
dataset

## 3. Basismodell laden (4-Bit, QLoRA)

Die eingefrorene Basis wird auf 4-Bit (NF4) komprimiert geladen - das ist das "Q" in QLoRA. Siehe das
LoRA-Schaltplan-Artifact fuer die Begruendung.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Kein manuelles tokenizer.pad_token = eos_token mehr (Lauf 1): trl setzt den
# Pad-Token selbst, und Qwens eigener Pad-Token (<|endoftext|>) ist definiert.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

## 4. LoRA-Adapter aufsetzen

Lauf 2: `target_modules="all-linear"` statt nur `q/k/v/o_proj` - also auch die MLP-Schichten
(`gate_proj`/`up_proj`/`down_proj`). Lauf 1 zeigte Underfitting, nicht Overfitting, daher mehr Kapazitaet
statt weniger. `print_trainable_parameters()` zeigt gleich die echte Zahl (Erwartung: ca. 18,5 Mio., ~1,2 %
des Modells; Lauf 1 hatte ca. 4,4 Mio.).

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Daten ins Prompt/Completion-Format bringen

Jede Zeile in `train.jsonl`/`val.jsonl` hat die Form `{"messages": [user, assistant], ...}` (siehe
`training/data_gen/dataset.py`). Fuer Lauf 2 wird sie in `{"prompt": [user], "completion": [assistant]}` zerlegt:
Nur bei diesem Format setzt `trl` den Loss automatisch ausschliesslich auf die Antwort (`completion_only_loss`)
und wendet Qwens Chat-Template selbst an - die manuelle `apply_chat_template`-Zelle aus Lauf 1 entfaellt.

In [ ]:
def to_prompt_completion(example):
    user_message, assistant_message = example["messages"]
    return {"prompt": [user_message], "completion": [assistant_message]}


formatted_dataset = dataset.map(
    to_prompt_completion, remove_columns=dataset["train"].column_names
)
print(formatted_dataset["train"][0])

## 6. Training

`EarlyStoppingCallback` bricht ab, sobald der Val-Loss zwei Epochen in Folge nicht mehr sinkt - die Bremse
gegen Auswendiglernen bei nur 217 Trainingsbeispielen. In Lauf 1 (Geduld 1, 3 Epochen) griff sie nie; jetzt
sind bis zu 5 Epochen erlaubt (14 Schritte pro Epoche, ~70 Schritte insgesamt, grob 20-30 Minuten auf der T4).

`completion_only_loss=True` ist bei Prompt/Completion-Daten ohnehin der Default und steht hier zur
Deutlichkeit explizit. `warmup_steps=3` entspricht wieder ~4 % der Schritte.

Falls `SFTConfig`/`SFTTrainer` sich seit Schreiben dieses Notebooks in `trl` geaendert haben, zeigt der Fehler
meist direkt, welcher Parameter umbenannt wurde (geschrieben gegen trl 1.13.0).

In [ ]:
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="qwen2.5-1.5b-enemy-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=5,
    completion_only_loss=True,
    max_length=512,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.train()

## 7. Sofort speichern

Direkt nach dem Training, bevor irgendwas anderes laeuft: Colab trennt Laufzeiten nach einer Weile Inaktivitaet
von selbst, und dabei geht **alles im Speicher verloren**, inklusive des gerade trainierten Modells - es existiert
bis hierhin nur im GPU-/RAM-Speicher dieser Sitzung, nicht auf Platte. Der Sanity-Check danach ist informativ,
aber optional; das Speichern hier nicht.

Zwei Varianten: der winzige Adapter allein (portabel, wenige MB) und das gemergte Vollmodell (Basis + Adapter
zusammengefuehrt zu normalen Gewichten) - Letzteres brauchen wir in Schritt 7 des Gesamtprojekts fuer die
GGUF-Quantisierung, da llama.cpp/Ollama ein zusammenhaengendes Modell erwartet, nicht Basis+Adapter getrennt.

**Wichtig, per echtem Fehlschlag gelernt**: `merge_and_unload()` direkt auf dem 4-Bit-QLoRA-Trainingsmodell
aufzurufen, dequantisiert nicht zuverlaessig - das Ergebnis kann intern 4-Bit bleiben (erkennbar an
`quantization_config` in der gespeicherten `config.json` und an einer Dateigroesse, die viel zu klein fuer
echtes bf16 ist). Robuster: das Basismodell frisch in voller Praezision laden, den (bereits gespeicherten)
Adapter draufsetzen, dann erst mergen - das umgeht das Dequantisierungsproblem komplett.

In [ ]:
import gc
import os

from peft import PeftModel

ADAPTER_DIR = "qwen2.5-1.5b-enemy-v2-lora-adapter"
MERGED_DIR = "qwen2.5-1.5b-enemy-v2-merged"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Reload the base model in full precision (NOT the 4-bit bnb_config from step 3) and
# apply the adapter on top of that clean copy, instead of merging out of the 4-bit
# training model directly - see the note above for why.
del model
gc.collect()
torch.cuda.empty_cache()

base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto"
)  # older transformers: use torch_dtype= instead of dtype= if this errors
peft_model = PeftModel.from_pretrained(base_model_fp16, ADAPTER_DIR)
merged_model = peft_model.merge_and_unload()

# Belt and suspenders: make sure no stale 4-bit quantization_config survives into the
# saved config.json even if a future transformers version carries one over.
merged_model.config.quantization_config = None

merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

size_gb = sum(os.path.getsize(os.path.join(MERGED_DIR, f)) for f in os.listdir(MERGED_DIR)) / 1e9
print(f"Merged model on disk: {size_gb:.2f} GB (erwartet: ~3 GB fuer bf16, NICHT ~1.6 GB)")

## 8. Schnelltest: sieht das vernuenftig aus, und wie oft ist es gueltig?

Kein Ersatz fuer den Eval-Harness aus Schritt 6 - aber ein schneller Go/No-Go, bevor du 2,4 GB herunterlaedst.
Erst drei Beispiele zum Anschauen, dann der Anteil gueltiger Antworten (parsebares JSON **mit allen fuenf
Feldern**) ueber alle 40 Val-Prompts, gesampelt bei Temperatur 0.9 wie in Produktion. Das eigentlich Wichtige
(das trainierte Modell) ist an dieser Stelle schon gespeichert - hier steht nichts mehr auf dem Spiel.

Vergleichswerte lokal gemessen: Lauf 1 ca. 23-36 %, Groq ca. 99 %, Basismodell 0 %. Beachte: das hier ist das
bf16-Modell direkt in Colab, ohne Ollama/Quantisierung.

In [ ]:
import json

merged_model.eval()
REQUIRED_FIELDS = {"name", "description", "hp", "attack", "attack_name"}


def generate_reply(user_prompt):
    chat = [{"role": "user", "content": user_prompt}]
    input_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(merged_model.device)
    output = merged_model.generate(
        **inputs, max_new_tokens=150, temperature=0.9, do_sample=True
    )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)


for example in dataset["validation"].select(range(3)):
    user_prompt = example["messages"][0]["content"]
    print("PROMPT:", user_prompt[:100], "...")
    print("GENERIERT:", generate_reply(user_prompt))
    print("ERWARTET (Groq):", example["messages"][1]["content"])
    print("---")

valid = 0
parsed_but_no_attack_name = 0
for example in dataset["validation"]:
    reply = generate_reply(example["messages"][0]["content"])
    try:
        parsed = json.loads(reply)
    except json.JSONDecodeError:
        continue
    if not isinstance(parsed, dict):
        continue
    valid += REQUIRED_FIELDS <= set(parsed)
    parsed_but_no_attack_name += "attack_name" not in parsed

total = len(dataset["validation"])
print(f"Gueltig (JSON + alle 5 Felder): {valid}/{total} = {valid / total:.0%}")
print(f"Parsebares JSON, aber ohne attack_name: {parsed_but_no_attack_name}/{total}")

## 9. Herunterladen

Bewusst erst **nach** dem Schnelltest: ein Zip+Download des ~3 GB grossen Modells lohnt sich nur, wenn der Anteil
gueltiger Antworten oben deutlich besser ist als in Lauf 1. Beide Modell-Ordner liegen ohnehin schon auf der
Colab-VM (Schritt 7) - du kannst diese Zelle auch spaeter noch ausfuehren, solange die Sitzung lebt.

In [ ]:
!zip -r qwen2.5-1.5b-enemy-v2-merged.zip qwen2.5-1.5b-enemy-v2-merged
!zip -r qwen2.5-1.5b-enemy-v2-lora-adapter.zip qwen2.5-1.5b-enemy-v2-lora-adapter

from google.colab import files

files.download("qwen2.5-1.5b-enemy-v2-merged.zip")
files.download("qwen2.5-1.5b-enemy-v2-lora-adapter.zip")